# Lab 01-B — Provedores: GPT, Gemini e Opus

**Skill & GO · Agentic Engineering · Aula 01 — Fundamentos**

Neste lab fazemos a **mesma chamada simples** a três provedores de IA as a Service e observamos, em cada um:

- **Dados:** o texto da resposta;
- **Metadados:** as informações que o provedor devolve junto com a resposta;
- **Janela de contexto:** quantos tokens o modelo aceita;
- **Custo:** quanto a chamada custou, a partir dos tokens usados.

## 1. Preparar o ambiente

Instalamos o LangChain e as integrações dos três provedores.

In [ ]:
%pip install -qU "langchain>=1.4,<2" "langchain-openai>=1.6,<2" "langchain-google-genai>=4.4,<5" "langchain-anthropic>=1.7,<2"

Agora as chaves. No Colab, cadastre no painel **🔑 Secrets** as chaves que você tiver — `OPENAI_API_KEY`, `GOOGLE_API_KEY` e `ANTHROPIC_API_KEY` — e habilite o acesso para este notebook.
Provedor sem chave é pulado.

In [ ]:
import logging
import os
from getpass import getpass

logging.getLogger("google_genai.models").setLevel(logging.ERROR)  # esconde um aviso interno do SDK do Gemini

for nome in ["OPENAI_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY"]:
    try:
        from google.colab import userdata
        os.environ[nome] = userdata.get(nome)
    except ImportError:
        pass  # fora do Colab: usa a variável de ambiente
    except Exception:
        pass  # segredo não cadastrado no Colab

    if not os.environ.get(nome):
        os.environ[nome] = getpass(f"Cole sua {nome} (ou Enter para pular): ")

    print(nome, "✅" if os.environ[nome] else "⚠️ ausente — este provedor será pulado")

## 2. Modelos, preços e pergunta

Os preços estão em **US$ por 1 milhão de tokens** e foram consultados em setembro de 2026 nas páginas oficiais:
[OpenAI](https://developers.openai.com/api/docs/models/gpt-5.6-luna),
[Google](https://ai.google.dev/gemini-api/docs/pricing) e
[Anthropic](https://platform.claude.com/docs/en/about-claude/pricing).
Preços mudam com frequência — confira antes de usar em produção.

In [ ]:
MODELOS = {
    "GPT": {"id": "openai:gpt-5.6-luna", "chave": "OPENAI_API_KEY", "entrada": 0.20, "saida": 1.20},
    "Gemini": {"id": "google_genai:gemini-3.7-flash", "chave": "GOOGLE_API_KEY", "entrada": 0.75, "saida": 3.75},  # até 31/12/2026
    "Opus": {"id": "anthropic:claude-opus-5", "chave": "ANTHROPIC_API_KEY", "entrada": 5.00, "saida": 25.00},
}

PERGUNTA = "Em no máximo duas frases, explique o que é a janela de contexto de um LLM."

## 3. A chamada simples

A função `chamar` faz a mesma coisa para qualquer provedor:

1. cria o modelo com `init_chat_model`;
2. envia a pergunta com `invoke`;
3. mostra **dados**, **metadados**, **janela de contexto** e **custo**.

A janela de contexto vem do `profile` do modelo — a ficha técnica que o LangChain mantém para cada modelo conhecido.

In [ ]:
from pprint import pprint

from langchain.chat_models import init_chat_model


def chamar(provedor: str):
    config = MODELOS[provedor]
    if not os.environ.get(config["chave"]):
        print(f"⚠️ {config['chave']} não configurada — pulando o {provedor}.")
        return None

    modelo = init_chat_model(config["id"])
    resposta = modelo.invoke(PERGUNTA)

    print("📄 DADOS — texto da resposta")
    print(resposta.text)

    print("\n🏷️ METADADOS — response_metadata")
    pprint(resposta.response_metadata)

    print("\n🔢 TOKENS — usage_metadata")
    uso = resposta.usage_metadata
    pprint(uso)

    perfil = modelo.profile or {}
    entrada, saida = perfil.get("max_input_tokens"), perfil.get("max_output_tokens")
    print("\n🪟 JANELA DE CONTEXTO")
    print(f"{entrada:,} tokens de entrada | {saida:,} tokens de saída" if entrada else "Modelo sem ficha técnica no LangChain.")

    custo = (uso["input_tokens"] * config["entrada"] + uso["output_tokens"] * config["saida"]) / 1_000_000
    print("\n💰 CUSTO")
    print(f"US$ {custo:.6f} nesta chamada (≈ US$ {custo * 1000:.2f} a cada 1.000 chamadas iguais)")

    return {
        "provedor": provedor,
        "modelo": config["id"],
        "janela de contexto": entrada,
        "tokens de entrada": uso["input_tokens"],
        "tokens de saída": uso["output_tokens"],
        "custo (US$)": round(custo, 6),
    }

## 4. GPT (OpenAI)

Se aparecer `reasoning` em `output_token_details`, são tokens que o modelo usou "pensando" antes de responder — e eles são cobrados como saída.

In [ ]:
gpt = chamar("GPT")

## 5. Gemini (Google)

Compare os campos de `response_metadata` com os do GPT: cada provedor devolve metadados diferentes.

In [ ]:
gemini = chamar("Gemini")

## 6. Opus (Anthropic)

O Opus 5 decide sozinho quanto pensar antes de responder (*adaptive thinking*); esse raciocínio também entra nos tokens de saída.

In [ ]:
opus = chamar("Opus")

## 7. Comparando os provedores

Observe na tabela:

- a mesma pergunta vira quantidades **diferentes** de tokens de entrada, porque cada provedor tem seu próprio tokenizador;
- os tokens de saída incluem o raciocínio dos modelos que pensam antes de responder;
- o custo varia muito de um modelo para outro, e a janela de contexto limita quanto texto cabe em uma única chamada.

In [ ]:
import pandas as pd

pd.DataFrame([resultado for resultado in [gpt, gemini, opus] if resultado])